# Notebook 11 — Kenapa Model Sulit Belajar Pola dari Data OHLCV + Sentimen?

**Masalah:** model LSTM (fitur OHLCV-return + `sentiment_mean`, horizon prediksi 5 hari)
performanya mendekati tebak acak — akurasi/F1 tidak jauh beda dari melempar koin.
Notebook ini mencari tahu **APAKAH ini masalah datanya (sinyal memang lemah/tidak ada)
atau masalah modelnya (arsitektur/training kurang tepat)**, lewat 5 sudut pandang
statistik yang saling melengkapi — semuanya dijelaskan dalam bahasa sederhana supaya
bisa langsung dipakai laporan ke dosen pembimbing.

## Peta 5 Metrik di Notebook Ini

| # | Metrik | Pertanyaan yang dijawab |
|---|--------|--------------------------|
| 1 | **Keseimbangan label** | Apakah kelas Naik/Turun seimbang? (kalau timpang, akurasi tinggi bisa menipu) |
| 2 | **Korelasi Point-Biserial** | Apakah nilai fitur hari ini berhubungan **linear** dengan arah harga nanti? |
| 3 | **Mutual Information** | Kalau bukan hubungan linear — apakah ada hubungan **apa pun** (linear atau tidak)? |
| 4 | **Autokorelasi Return** | Apakah pola harga masa lalu bisa dipakai menebak pola harga masa depan? |
| 5 | **Effect Size (Cohen's d)** | Seberapa jauh "jarak" rata-rata nilai fitur antara hari-Naik vs hari-Turun? |
| 6 | **Uji model paling sederhana** | Kalau pakai model paling sederhana (garis lurus pemisah), bisakah mengalahkan tebak acak? |

**Prinsip:** kalau **SEMUA** metrik di atas menunjukkan "nyaris tidak ada hubungan", itu
bukti kuat kalau masalahnya ada di **datanya** (sinyal memang sangat lemah/tidak ada) —
bukan salah arsitektur LSTM. Sebaliknya, kalau ada metrik yang menunjukkan hubungan
tapi model tetap gagal belajar, baru itu petunjuk masalah ada di sisi model/training.

Fitur & konfigurasi yang dianalisis **PERSIS SAMA** dengan yang dipakai training resmi
(`app/services/training_service.py` → `models/lstm/notebook07_common.py`) — fitur
**return** (`pct_change` OHLCV, BUKAN harga level) + `sentiment_mean`, horizon 5 hari,
window 20 hari, split kronologis 70/15/15 — supaya diagnosis ini benar-benar
mencerminkan kondisi model yang sedang dikeluhkan, bukan simulasi terpisah.

## Setup & Muat Data

Reuse langsung fungsi produksi — TIDAK reimplementasi:
- `load_return_features()` + `RETURN_FEATURE_COLS_A/B` (`dataset_builder.py`) — fitur return OHLCV + sentimen.
- `load_split_capped()` (`notebook07_common.py`) — split kronologis 70/15/15 + capping outlier IQR (fit-train-only), SAMA PERSIS logic yang dipakai training resmi.
- `prepare_dataset_bundle()` (`notebook07_common.py`) — dipakai khusus di Bagian 6 (scaling + windowing identik training).

In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
if (PROJECT_ROOT / "pipeline").exists() is False:
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

from pipeline.feature_engineering.dataset_builder import (
    RETURN_FEATURE_COLS_A,
    RETURN_FEATURE_COLS_B,
    load_return_features,
)
from models.lstm.notebook07_common import load_split_capped, prepare_dataset_bundle
from models.lstm.notebook07_trainer import RETURN_HORIZON, WINDOW

TRAIN_RATIO, VAL_RATIO = 0.70, 0.15  # sama seperti Random Search resmi

pd.set_option("display.float_format", lambda v: f"{v:.4f}")

print(f"Fitur Model B (gabungan)  : {RETURN_FEATURE_COLS_B}")
print(f"Fitur Model A (OHLCV saja): {RETURN_FEATURE_COLS_A}")
print(f"Horizon prediksi: {RETURN_HORIZON} hari | Window LSTM: {WINDOW} hari | Split: {TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{1-TRAIN_RATIO-VAL_RATIO:.0%}")

Fitur Model B (gabungan)  : ['ret_open', 'ret_high', 'ret_low', 'ret_close', 'ret_volume', 'sentiment_mean']
Fitur Model A (OHLCV saja): ['ret_open', 'ret_high', 'ret_low', 'ret_close', 'ret_volume']
Horizon prediksi: 5 hari | Window LSTM: 20 hari | Split: 70%/15%/15%


In [17]:
df_train, df_val, df_test, capping_bounds, n_capped = load_split_capped(
    load_return_features, RETURN_FEATURE_COLS_B, horizon=RETURN_HORIZON,
    train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
)
print(f"Baris — train: {len(df_train)} | val: {len(df_val)} | test: {len(df_test)}")
df_train.head()

Baris — train: 1064 | val: 228 | test: 229


,Date,ret_open,ret_high,ret_low,ret_close,ret_volume,sentiment_mean,Close,label
0,2018-01-03,0.0053,0.0069,0.0057,0.0064,0.0432,-0.1667,2713.0601,1
1,2018-01-04,0.0080,0.0055,0.0079,0.0040,0.0433,0.0000,2723.9900,1
2,2018-01-05,0.0044,0.0052,0.0033,0.0070,-0.1239,0.0000,2743.1499,1
3,2018-01-08,0.0042,0.0018,0.0035,0.0017,0.0021,0.1667,2747.7100,1
4,2018-01-09,0.0031,0.0039,0.0037,0.0013,0.0682,0.6667,2751.2900,1


## Bagian 1 — Apakah Label (Naik/Turun) Seimbang?

**Kenapa ini dicek duluan:** kalau satu kelas jauh lebih banyak dari yang lain (misalnya
Naik 70% vs Turun 30%), model yang MALAS dan cuma menebak "Naik" terus-menerus akan
kelihatan "akurat" 70% — padahal dia sama sekali tidak belajar apa-apa. Jadi sebelum
menilai model dari angka akurasi, kita harus tahu dulu berapa akurasi yang bisa dicapai
HANYA DENGAN menebak kelas mayoritas terus-menerus (baseline paling malas).

In [18]:
for name, split_df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    if len(split_df) == 0:
        print(f"{name:5s}: (kosong)")
        continue
    up_pct = split_df["label"].mean() * 100
    majority_baseline = max(up_pct, 100 - up_pct)
    print(f"{name:5s}: Naik={up_pct:5.1f}% | Turun={100-up_pct:5.1f}%  "
          f"-> akurasi baseline 'tebak kelas mayoritas terus': {majority_baseline:.1f}%")

print(
    "\nInterpretasi: kalau nanti akurasi model HANYA sedikit di atas (atau malah di bawah) "
    "angka baseline di atas, artinya model TIDAK benar-benar belajar pola apa pun — dia "
    "cuma condong ke kelas mayoritas (atau lebih buruk dari itu)."
)

train: Naik= 61.7% | Turun= 38.3%  -> akurasi baseline 'tebak kelas mayoritas terus': 61.7%
val  : Naik= 50.9% | Turun= 49.1%  -> akurasi baseline 'tebak kelas mayoritas terus': 50.9%
test : Naik= 65.5% | Turun= 34.5%  -> akurasi baseline 'tebak kelas mayoritas terus': 65.5%

Interpretasi: kalau nanti akurasi model HANYA sedikit di atas (atau malah di bawah) angka baseline di atas, artinya model TIDAK benar-benar belajar pola apa pun — dia cuma condong ke kelas mayoritas (atau lebih buruk dari itu).


## Bagian 2 — Korelasi Fitur vs Label (Point-Biserial)

**Apa ini:** versi korelasi Pearson khusus ketika salah satu variabelnya berupa
kategori 2 kelas (di sini: label Naik=1/Turun=0). Nilainya -1 s.d. +1, sama seperti
Pearson biasa — mengukur **hubungan garis lurus (linear)** antara nilai fitur hari
ini dengan arah harga 5 hari ke depan.

**Cara baca:** `|r|` mendekati 0 = **hampir tidak ada hubungan linear** yang bisa
dimanfaatkan model. Dihitung terpisah di train/val/test — kalau tandanya (+/-)
berubah-ubah antar split, itu tanda tambahan bahwa hubungannya tidak stabil/bisa
diandalkan (bukan pola asli, cuma kebetulan).

In [19]:
def interpret_r(r: float) -> str:
    a = abs(r)
    if a < 0.05:
        return "nyaris tidak ada hubungan"
    if a < 0.1:
        return "sangat lemah"
    if a < 0.3:
        return "lemah"
    if a < 0.5:
        return "sedang"
    return "kuat"


corr_rows = []
for split_name, split_df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    row = {"split": split_name}
    for col in RETURN_FEATURE_COLS_B:
        r, _p = stats.pointbiserialr(split_df["label"], split_df[col])
        row[col] = round(r, 4)
    corr_rows.append(row)
corr_df = pd.DataFrame(corr_rows).set_index("split")
print("=== Korelasi Point-Biserial (fitur vs label), per split ===")
print(corr_df)

print("\n=== Interpretasi (berdasar rata-rata |r| ketiga split) ===")
for col in RETURN_FEATURE_COLS_B:
    mean_abs_r = corr_df[col].abs().mean()
    signs = set(np.sign(corr_df[col].values))
    flip_note = "  [!] arah hubungan BERUBAH antar split (train/val/test)" if len(signs - {0}) > 1 else ""
    print(f"  {col:15s}: rata-rata |r|={mean_abs_r:.4f} -> {interpret_r(mean_abs_r)}{flip_note}")

=== Korelasi Point-Biserial (fitur vs label), per split ===
       ret_open  ret_high  ret_low  ret_close  ret_volume  sentiment_mean
split                                                                    
train    0.0014   -0.0204   0.0071    -0.0170     -0.0321          0.0262
val     -0.0525   -0.0759  -0.0369    -0.0812      0.0379         -0.0836
test     0.0271    0.0252   0.0536     0.0515     -0.0567          0.0859

=== Interpretasi (berdasar rata-rata |r| ketiga split) ===
  ret_open       : rata-rata |r|=0.0270 -> nyaris tidak ada hubungan  [!] arah hubungan BERUBAH antar split (train/val/test)
  ret_high       : rata-rata |r|=0.0405 -> nyaris tidak ada hubungan  [!] arah hubungan BERUBAH antar split (train/val/test)
  ret_low        : rata-rata |r|=0.0325 -> nyaris tidak ada hubungan  [!] arah hubungan BERUBAH antar split (train/val/test)
  ret_close      : rata-rata |r|=0.0499 -> nyaris tidak ada hubungan  [!] arah hubungan BERUBAH antar split (train/val/test)
  ret_volu

## Bagian 3 — Mutual Information (Hubungan Non-Linear)

**Kenapa perlu ini:** korelasi di Bagian 2 CUMA bisa menangkap hubungan **garis
lurus**. Bisa saja ada hubungan yang bentuknya bukan garis lurus (misalnya
"kalau sentimen SANGAT negatif ATAU SANGAT positif, harga cenderung turun, tapi
kalau netral tidak ada pola") — korelasi Pearson/point-biserial akan menyebutnya
"tidak ada hubungan" padahal sebenarnya ada, hanya bentuknya bukan garis lurus.

**Mutual Information (MI)** mengukur hubungan APA PUN (linear maupun tidak) antara
fitur dan label. Nilainya mulai dari 0 (fitur & label benar-benar tidak saling
memberi info) dan makin besar kalau makin ada ketergantungan.

**Cara baca gabungan dengan Bagian 2:** kalau korelasi RENDAH tapi MI juga RENDAH →
benar-benar hampir tidak ada hubungan apa pun (linear ataupun tidak). Kalau korelasi
rendah tapi MI relatif TINGGI (dibanding fitur lain) → ada hubungan non-linear yang
berpotensi bisa ditangkap model yang lebih fleksibel seperti LSTM.

In [20]:
mi_scores = mutual_info_classif(
    df_train[RETURN_FEATURE_COLS_B].values, df_train["label"].values,
    discrete_features=False, random_state=42,
)
mi_df = pd.DataFrame({"fitur": RETURN_FEATURE_COLS_B, "mutual_info": mi_scores})
mi_df["korelasi_|r|_train"] = [abs(corr_df.loc["train", c]) for c in RETURN_FEATURE_COLS_B]
mi_df = mi_df.sort_values("mutual_info", ascending=False).reset_index(drop=True)

print("=== Mutual Information vs Korelasi (data train) ===")
print(mi_df)

if mi_df["mutual_info"].max() < 0.01:
    print(
        "\nInterpretasi: SEMUA nilai Mutual Information mendekati 0 -> tidak ada hubungan "
        "yang terdeteksi antara fitur dan label, baik linear (Bagian 2) MAUPUN non-linear. "
        "Ini memperkuat dugaan sinyalnya memang sangat lemah di data, bukan cuma soal "
        "bentuk hubungannya yang tidak linear."
    )
else:
    top_feat = mi_df.iloc[0]["fitur"]
    print(
        f"\nInterpretasi: fitur '{top_feat}' punya Mutual Information relatif TERTINGGI -> "
        "ada indikasi hubungan non-linear (mungkin tidak tertangkap korelasi Pearson di "
        "Bagian 2) yang berpotensi bisa dipelajari model yang lebih fleksibel."
    )

=== Mutual Information vs Korelasi (data train) ===
            fitur  mutual_info  korelasi_|r|_train
0      ret_volume       0.0173              0.0321
1        ret_open       0.0141              0.0014
2       ret_close       0.0134              0.0170
3         ret_low       0.0086              0.0071
4        ret_high       0.0000              0.0204
5  sentiment_mean       0.0000              0.0262

Interpretasi: fitur 'ret_volume' punya Mutual Information relatif TERTINGGI -> ada indikasi hubungan non-linear (mungkin tidak tertangkap korelasi Pearson di Bagian 2) yang berpotensi bisa dipelajari model yang lebih fleksibel.


## Bagian 4 — Autokorelasi Return: Apakah Pola Harga Masa Lalu Bisa Menebak Masa Depan?

**Ide dasar:** kalau harga PUNYA pola berulang (misalnya "kalau naik 3 hari
berturut-turut, biasanya lanjut naik lagi"), maka return hari ini seharusnya
berkorelasi dengan return beberapa hari sebelumnya. Ini disebut **autokorelasi**
— korelasi sebuah deret angka dengan versi "mundur" dari dirinya sendiri (lag).

**Kenapa penting untuk kasus ini:** ini adalah pertanyaan yang LEBIH MENDASAR
daripada "apakah model bisa belajar" — ini tentang apakah harga saham secara
STATISTIK memang punya pola yang bisa diulang. Kalau autokorelasinya mendekati 0
di semua lag, itu konsisten dengan **hipotesis pasar efisien** (Efficient Market
Hypothesis) — gagasan yang sudah lama dikenal di ilmu keuangan bahwa harga saham
bergerak mendekati acak (random walk) karena semua informasi yang tersedia sudah
"diserap" ke harga saat ini. Kalau ini yang terjadi, **model APA PUN — bukan cuma
LSTM — akan kesulitan** memprediksi arah harga dari pola harga masa lalu saja.

In [21]:
df_full = load_return_features()
LAGS = [1, 2, 3, 4, 5]

print("=== Autokorelasi ret_close pada lag 1-5 hari (seluruh data) ===")
autocorr_rows = []
for lag in LAGS:
    ac = df_full["ret_close"].autocorr(lag=lag)
    autocorr_rows.append({"lag_hari": lag, "autokorelasi": round(ac, 4)})
    print(f"  lag {lag} hari: {ac:+.4f}")

autocorr_df = pd.DataFrame(autocorr_rows)
max_abs_ac = autocorr_df["autokorelasi"].abs().max()
if max_abs_ac < 0.05:
    print(
        "\nInterpretasi: autokorelasi di SEMUA lag mendekati 0 -> return harga hari ini "
        "praktis TIDAK berhubungan dengan return beberapa hari sebelumnya. Ini sejalan "
        "dengan hipotesis pasar efisien (harga bergerak mendekati acak/random walk) -- "
        "temuan umum di literatur keuangan untuk data return harian, bukan cuma masalah "
        "khusus di proyek ini."
    )
else:
    print(
        f"\nInterpretasi: ada lag dengan autokorelasi cukup terlihat (maks |r|={max_abs_ac:.3f}) "
        "-> ada indikasi pola berulang di pergerakan harga yang berpotensi bisa dimanfaatkan."
    )

=== Autokorelasi ret_close pada lag 1-5 hari (seluruh data) ===
  lag 1 hari: -0.1678
  lag 2 hari: +0.1017
  lag 3 hari: -0.0199
  lag 4 hari: -0.0571
  lag 5 hari: +0.0552

Interpretasi: ada lag dengan autokorelasi cukup terlihat (maks |r|=0.168) -> ada indikasi pola berulang di pergerakan harga yang berpotensi bisa dimanfaatkan.


## Bagian 5 — Effect Size (Cohen's d): Seberapa "Jauh" Fitur Membedakan Hari Naik vs Turun?

**Kenapa perlu ini di samping korelasi:** korelasi itu abstrak (angka -1 s.d. 1),
sedangkan **Cohen's d** lebih mudah dibayangkan — dia mengukur **jarak rata-rata**
nilai fitur antara kelompok "hari yang akan Naik" vs "hari yang akan Turun",
dibagi seberapa "lebar" sebaran datanya. Analoginya: kalau tinggi rata-rata pria
170cm dan wanita 160cm dengan sebaran 10cm, itu beda yang cukup terlihat (d≈1). Tapi
kalau bedanya cuma 1cm dengan sebaran 10cm (d≈0.1), dua kelompok itu praktis
**tumpang tindih total** — tidak bisa dibedakan hanya dari satu angka itu saja.

**Aturan baca umum (Cohen):** `|d| < 0.2` = beda dapat diabaikan (tumpang tindih
total), `0.2–0.5` = beda kecil, `0.5–0.8` = beda sedang, `> 0.8` = beda besar.

In [22]:
def cohens_d(group_up: pd.Series, group_down: pd.Series) -> float:
    n1, n2 = len(group_up), len(group_down)
    pooled_std = np.sqrt(((n1 - 1) * group_up.std() ** 2 + (n2 - 1) * group_down.std() ** 2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0.0
    return (group_up.mean() - group_down.mean()) / pooled_std


def interpret_d(d: float) -> str:
    a = abs(d)
    if a < 0.2:
        return "dapat diabaikan (tumpang tindih total)"
    if a < 0.5:
        return "kecil"
    if a < 0.8:
        return "sedang"
    return "besar"


print("=== Cohen's d per fitur (data train): hari akan-Naik vs akan-Turun ===")
d_rows = []
for col in RETURN_FEATURE_COLS_B:
    grp_up = df_train.loc[df_train["label"] == 1, col]
    grp_down = df_train.loc[df_train["label"] == 0, col]
    d = cohens_d(grp_up, grp_down)
    d_rows.append({"fitur": col, "cohens_d": round(d, 4), "interpretasi": interpret_d(d)})
d_df = pd.DataFrame(d_rows).sort_values("cohens_d", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
print(d_df.to_string(index=False))

=== Cohen's d per fitur (data train): hari akan-Naik vs akan-Turun ===
         fitur  cohens_d                           interpretasi
    ret_volume   -0.0660 dapat diabaikan (tumpang tindih total)
sentiment_mean    0.0538 dapat diabaikan (tumpang tindih total)
      ret_high   -0.0419 dapat diabaikan (tumpang tindih total)
     ret_close   -0.0349 dapat diabaikan (tumpang tindih total)
       ret_low    0.0147 dapat diabaikan (tumpang tindih total)
      ret_open    0.0029 dapat diabaikan (tumpang tindih total)


## Bagian 6 — Uji Model Paling Sederhana: Bisakah Mengalahkan Tebak Acak?

**Ide:** LSTM adalah model yang KOMPLEKS — kalau dia gagal belajar, kita tidak
langsung tahu apakah itu karena datanya memang tidak punya pola, atau karena LSTM-nya
"kebingungan" oleh kerumitannya sendiri. Untuk memisahkan dua kemungkinan itu, kita
coba pakai model **paling sederhana yang ada**: Regresi Logistik (cuma menarik 1
garis/bidang pemisah lurus). Data yang dipakai (scaling + jendela 20 hari) **PERSIS
SAMA** dengan yang dilihat LSTM (`prepare_dataset_bundle()`, fungsi produksi) —
supaya perbandingannya adil.

Diuji 3 skema fitur:
1. **OHLCV saja** (`RETURN_FEATURE_COLS_A`, tanpa sentimen)
2. **Sentimen saja** (`sentiment_mean`)
3. **Gabungan** (`RETURN_FEATURE_COLS_B`, sama seperti Model B produksi)

**Cara baca AUC (Area Under Curve):** 0.50 = **persis sama dengan tebak koin**,
semakin jauh dari 0.50 (ke arah 1.0) semakin bagus. `<0.55` dianggap "praktis
tidak ada bedanya dengan tebak acak".

Kalau model sesederhana ini SAJA sudah tidak bisa mengalahkan tebak acak di
ketiga skema, itu **bukti kuat masalahnya ada di data (sinyal terlalu lemah)**,
bukan di kerumitan arsitektur LSTM — menambah lapisan/neuron LSTM tidak akan
menciptakan sinyal yang memang tidak ada di data.

In [23]:
def eval_simple_classifier(feature_cols: list[str], label: str) -> dict:
    bundle = prepare_dataset_bundle(
        load_data_fn=load_return_features,
        capping_cols=feature_cols,
        feature_cols=feature_cols,
        horizon=RETURN_HORIZON,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        window=WINDOW,
    )
    X_train_flat = bundle.X_train.reshape(len(bundle.X_train), -1)
    X_test_flat = bundle.X_test.reshape(len(bundle.X_test), -1)

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(X_train_flat, bundle.y_train)
    proba_test = clf.predict_proba(X_test_flat)[:, 1]
    pred_test = clf.predict(X_test_flat)

    auc = roc_auc_score(bundle.y_test, proba_test)
    acc = accuracy_score(bundle.y_test, pred_test)
    return {"skema": label, "n_fitur": len(feature_cols), "n_test": len(bundle.y_test), "auc_test": round(auc, 4), "akurasi_test": round(acc, 4)}


results_simple = [
    eval_simple_classifier(RETURN_FEATURE_COLS_A, "OHLCV saja"),
    eval_simple_classifier(["sentiment_mean"], "Sentimen saja"),
    eval_simple_classifier(RETURN_FEATURE_COLS_B, "Gabungan (OHLCV+sentimen)"),
]
results_simple_df = pd.DataFrame(results_simple)
print("=== Logistic Regression sederhana (fitur+jendela IDENTIK dgn input LSTM) ===")
print(results_simple_df.to_string(index=False))

print("\n=== Interpretasi per skema (AUC 0.50 = setara tebak acak) ===")
for row in results_simple:
    gap = row["auc_test"] - 0.5
    if abs(gap) < 0.05:
        verdict = "praktis TIDAK ADA bedanya dengan tebak acak"
    elif gap > 0:
        verdict = "ada sinyal, TAPI masih sangat lemah" if gap < 0.1 else "ada sinyal yang cukup terlihat"
    else:
        verdict = "LEBIH BURUK dari tebak acak (kemungkinan overfit ke pola yang salah arah)"
    print(f"  {row['skema']:28s}: AUC={row['auc_test']:.3f} -> {verdict}")

=== Logistic Regression sederhana (fitur+jendela IDENTIK dgn input LSTM) ===
                    skema  n_fitur  n_test  auc_test  akurasi_test
               OHLCV saja        5     210    0.5074        0.5095
            Sentimen saja        1     210    0.5988        0.5810
Gabungan (OHLCV+sentimen)        6     210    0.6004        0.5524

=== Interpretasi per skema (AUC 0.50 = setara tebak acak) ===
  OHLCV saja                  : AUC=0.507 -> praktis TIDAK ADA bedanya dengan tebak acak
  Sentimen saja               : AUC=0.599 -> ada sinyal, TAPI masih sangat lemah
  Gabungan (OHLCV+sentimen)   : AUC=0.600 -> ada sinyal yang cukup terlihat


## Ringkasan — Kartu Skor untuk Dosen Pembimbing

Sel di bawah menggabungkan hasil ke-6 bagian di atas jadi satu tabel & simpulan
singkat berbahasa sederhana, siap dipakai sebagai bahan diskusi.

In [24]:
print("=" * 78)
print("RINGKASAN DIAGNOSTIK -- KENAPA MODEL SULIT BELAJAR POLA")
print("=" * 78)

test_up_pct = df_test["label"].mean() * 100
imbalance_gap = abs(test_up_pct - 50)
mean_abs_r_test = corr_df.loc["test"].abs().mean()
max_mi = mi_df["mutual_info"].max()
max_autocorr = autocorr_df["autokorelasi"].abs().max()
max_d = d_df["cohens_d"].abs().max()

auc_ohlcv = results_simple_df.loc[results_simple_df["skema"] == "OHLCV saja", "auc_test"].iloc[0]
auc_sentiment = results_simple_df.loc[results_simple_df["skema"] == "Sentimen saja", "auc_test"].iloc[0]
auc_combined = results_simple_df.loc[results_simple_df["skema"] == "Gabungan (OHLCV+sentimen)", "auc_test"].iloc[0]

print(f"\n[1] Keseimbangan label (test): {test_up_pct:.1f}% Naik / {100 - test_up_pct:.1f}% Turun",
      "-> CUKUP TIMPANG, hati-hati baca akurasi mentah" if imbalance_gap > 10 else "-> relatif seimbang")
print(f"\n[2] Korelasi Point-Biserial (rata-rata |r| test, semua fitur): {mean_abs_r_test:.4f} -> {interpret_r(mean_abs_r_test)}")
print(f"\n[3] Mutual Information (maksimum, data train): {max_mi:.4f}",
      "-> tidak ada hubungan non-linear yang tertangkap" if max_mi < 0.01 else "-> ada indikasi hubungan non-linear (tapi masih sangat kecil)")
print(f"\n[4] Autokorelasi return (maksimum |r|, lag 1-5 hari): {max_autocorr:.4f}",
      "-> konsisten pasar efisien / random walk" if max_autocorr < 0.05 else "-> ada sedikit pola berulang, tapi masih lemah")
print(f"\n[5] Effect size Cohen's d (maksimum |d|, semua fitur): {max_d:.4f} -> {interpret_d(max_d)}")
print(f"\n[6] AUC Logistic Regression -- OHLCV saja: {auc_ohlcv:.3f} | Sentimen saja: {auc_sentiment:.3f} | Gabungan: {auc_combined:.3f} (0.50 = tebak acak)")

print("\n" + "=" * 78)
print("KESIMPULAN (dibentuk otomatis mengikuti angka di atas -- BUKAN narasi baku)")
print("=" * 78)

ohlcv_random = auc_ohlcv < 0.55
sentiment_has_signal = auc_sentiment >= 0.55
combined_has_signal = auc_combined >= 0.55

if ohlcv_random and not sentiment_has_signal and not combined_has_signal:
    print(
        "SEMUA skema (OHLCV saja, sentimen saja, gabungan) punya AUC mendekati 0.50 -> hampir "
        "tidak ada sinyal yang bisa dipelajari model APA PUN dari data ini untuk memprediksi "
        "arah harga 5 hari ke depan. Ini konsisten dengan hipotesis pasar efisien: informasi "
        "publik (harga historis & berita) sudah 'terserap' ke harga saat ini, sehingga sulit "
        "dipakai menebak arah ke depan. Implikasi utk skripsi: fokus laporan sebaiknya bukan "
        "'menaikkan akurasi model', melainkan mendokumentasikan dengan jujur seberapa lemah "
        "sinyal ini secara terukur -- itu sendiri adalah temuan ilmiah yang valid."
    )
elif ohlcv_random and (sentiment_has_signal or combined_has_signal):
    print(
        f"Fitur OHLCV SENDIRI praktis tidak mengandung sinyal (AUC={auc_ohlcv:.2f}, setara "
        "tebak acak) -- konsisten hipotesis pasar efisien: pola harga masa lalu (walau sudah "
        "diubah jadi return yang stasioner) tidak cukup untuk menebak arah harga ke depan. "
        f"TAPI fitur SENTIMEN berita menunjukkan sinyal kecil yang nyata (AUC={auc_sentiment:.2f}) "
        "dan independen dari harga. Artinya kesulitan model LSTM kemungkinan BUKAN 'data sama "
        "sekali tanpa pola', melainkan: (a) sinyal yang tersedia sangat KECIL (AUC~0.60 masih "
        "jauh dari model yang andal dipakai keputusan nyata), dan (b) menggabungkan 5 fitur "
        "OHLCV yang isinya nyaris cuma derau ke 1 fitur sentimen yang lemah berisiko "
        "'mengencerkan' sinyal kecil itu -- LSTM harus belajar mengabaikan mayoritas inputnya. "
        "Bahan diskusi ke dospem: pertimbangkan menguji/melaporkan varian model SENTIMEN-SAJA "
        "sebagai pembanding, dan tegaskan bahwa OHLCV terbukti BUKAN sumber sinyal utama di "
        "data ini (temuan ini sejalan dengan hasil awal notebook 13 sebelumnya)."
    )
else:
    print(
        f"Fitur OHLCV menunjukkan sinyal yang bisa dilihat (AUC={auc_ohlcv:.2f}), berbeda dari "
        "dugaan awal 'pasar sepenuhnya acak'. Kalau LSTM tetap gagal belajar padahal model "
        "sesederhana Regresi Logistik sudah menemukan sinyal ini, kemungkinan masalahnya ADA "
        "di sisi model/training (arsitektur, hyperparameter, atau cara split data) -- bukan "
        "murni keterbatasan data. Ini pola diagnosis yang berbeda dari dua kasus sebelumnya, "
        "cek ulang konfigurasi training (learning rate, window, class-weight, threshold)."
    )

print(
    f"\nCatatan tambahan: label test {'TIMPANG (' + f'{test_up_pct:.0f}%/{100 - test_up_pct:.0f}%' + ') -- kalau menilai model, WAJIB pakai F1-macro/AUC, jangan cuma akurasi mentah (bisa menipu).' if imbalance_gap > 10 else 'relatif seimbang, akurasi mentah cukup aman dipakai sbg salah satu metrik.'}"
)

RINGKASAN DIAGNOSTIK -- KENAPA MODEL SULIT BELAJAR POLA

[1] Keseimbangan label (test): 65.5% Naik / 34.5% Turun -> CUKUP TIMPANG, hati-hati baca akurasi mentah

[2] Korelasi Point-Biserial (rata-rata |r| test, semua fitur): 0.0500 -> nyaris tidak ada hubungan

[3] Mutual Information (maksimum, data train): 0.0173 -> ada indikasi hubungan non-linear (tapi masih sangat kecil)

[4] Autokorelasi return (maksimum |r|, lag 1-5 hari): 0.1678 -> ada sedikit pola berulang, tapi masih lemah

[5] Effect size Cohen's d (maksimum |d|, semua fitur): 0.0660 -> dapat diabaikan (tumpang tindih total)

[6] AUC Logistic Regression -- OHLCV saja: 0.507 | Sentimen saja: 0.599 | Gabungan: 0.600 (0.50 = tebak acak)

KESIMPULAN (dibentuk otomatis mengikuti angka di atas -- BUKAN narasi baku)
Fitur OHLCV SENDIRI praktis tidak mengandung sinyal (AUC=0.51, setara tebak acak) -- konsisten hipotesis pasar efisien: pola harga masa lalu (walau sudah diubah jadi return yang stasioner) tidak cukup untuk menebak arah 